In [1]:
# 完整的YOLO11模型转换流程：ONNX净化 + TensorFlow转换
# 用途：将YOLO11分类模型从ONNX格式转换为MTK APU兼容的TensorFlow PB格式

import onnx
from onnx import helper, numpy_helper, TensorProto
import numpy as np
import os
import subprocess
import sys

# ==================== 配置参数 ====================
INPUT_ONNX = "yolo11l-model.onnx"              # 输入的原始ONNX模型
SANITIZED_ONNX = "yolo11l-model_hsigmoid2clip.onnx"   # 净化后的ONNX模型
OUTPUT_PB = "yolo11l-model_fp32_frozen.pb"     # 最终的TensorFlow PB文件（匹配实际转换脚本）
CONVERSION_SCRIPT = "convert_onnx_to_pb.py"    # TensorFlow转换脚本

print("🚀 YOLO11模型完整转换流程开始")
print("=" * 60)

# ==================== 步骤1：检查输入文件 ====================
print("📋 步骤1/4: 检查输入文件...")

if not os.path.exists(INPUT_ONNX):
    print(f"❌ 错误：找不到输入ONNX模型 '{INPUT_ONNX}'")
    print("请确保已导出ONNX模型文件")
    exit(1)

file_size = os.path.getsize(INPUT_ONNX) / (1024*1024)
print(f"✅ 输入模型: {INPUT_ONNX} ({file_size:.2f} MB)")

# 检查原始模型的算子
original_model = onnx.load(INPUT_ONNX)
original_ops = {}
for node in original_model.graph.node:
    original_ops[node.op_type] = original_ops.get(node.op_type, 0) + 1

print(f"原始模型算子: {dict(sorted(original_ops.items()))}")
hsigmoid_count = original_ops.get('HardSigmoid', 0)
print(f"需要处理的HardSigmoid节点: {hsigmoid_count}个")

if hsigmoid_count == 0:
    print("⚠️  模型中没有HardSigmoid节点，但仍执行净化流程以确保兼容性")

# ==================== 步骤2：ONNX净化处理 ====================
print(f"\n🔧 步骤2/4: ONNX净化处理 (HardSigmoid → Mul+Add+Clip)...")

try:
    model = onnx.load(INPUT_ONNX)
    graph = model.graph
    
    # 构建消费者映射
    consumers = {}
    for node in graph.node:
        for inp in node.input:
            if inp:  # 忽略空输入
                consumers.setdefault(inp, []).append(node)
    
    # 创建新的节点列表
    new_node_list = []
    new_initializers = []
    replacement_map = {}  # 映射原输出到新输出
    processed_count = 0
    
    for i, node in enumerate(graph.node):
        if node.op_type != "HardSigmoid":
            # 非HardSigmoid节点，更新输入引用后添加
            updated_inputs = [replacement_map.get(inp, inp) for inp in node.input]
            new_node = helper.make_node(
                node.op_type,
                updated_inputs,
                list(node.output),
                name=node.name,
                **{attr.name: helper.get_attribute_value(attr) for attr in node.attribute}
            )
            new_node_list.append(new_node)
        else:
            # HardSigmoid节点替换
            x = node.input[0]
            y = node.output[0]
            
            # 获取HardSigmoid参数
            alpha = 0.2  # ONNX默认值
            beta = 0.5   # ONNX默认值
            for attr in node.attribute:
                if attr.name == "alpha":
                    alpha = attr.f
                elif attr.name == "beta":
                    beta = attr.f
            
            print(f"  处理节点 {processed_count+1}/{hsigmoid_count}: {node.name or f'HardSigmoid_{i}'} (α={alpha}, β={beta})")
            
            # 生成唯一名称
            idx = processed_count
            alpha_name = f"const_alpha_{idx}"
            beta_name = f"const_beta_{idx}"
            mul_out = f"mul_alpha_out_{idx}"
            add_out = f"add_beta_out_{idx}"
            clip_out = f"clip_out_{idx}"
            
            # 创建常量
            alpha_init = numpy_helper.from_array(np.array(alpha, dtype=np.float32), name=alpha_name)
            beta_init = numpy_helper.from_array(np.array(beta, dtype=np.float32), name=beta_name)
            new_initializers.extend([alpha_init, beta_init])
            
            # 创建替换序列：input → Mul(α) → Add(β) → Clip(0,1)
            actual_x = replacement_map.get(x, x)
            
            mul_node = helper.make_node("Mul", [actual_x, alpha_name], [mul_out], name=f"Mul_Alpha_{idx}")
            add_node = helper.make_node("Add", [mul_out, beta_name], [add_out], name=f"Add_Beta_{idx}")
            clip_node = helper.make_node("Clip", [add_out], [clip_out], name=f"Clip_{idx}", min=0.0, max=1.0)
            
            new_node_list.extend([mul_node, add_node, clip_node])
            replacement_map[y] = clip_out
            processed_count += 1
    
    # 重建图
    del graph.node[:]
    graph.node.extend(new_node_list)
    graph.initializer.extend(new_initializers)
    
    # 可选形状推断
    try:
        model = onnx.shape_inference.infer_shapes(model)
        print("  ✅ 形状推断完成")
    except Exception as e:
        print(f"  ⚠️  形状推断警告: {e}")
    
    # 保存净化后的模型
    onnx.save(model, SANITIZED_ONNX)
    
    # 验证结果
    sanitized_model = onnx.load(SANITIZED_ONNX)
    sanitized_ops = {}
    for node in sanitized_model.graph.node:
        sanitized_ops[node.op_type] = sanitized_ops.get(node.op_type, 0) + 1
    
    sanitized_size = os.path.getsize(SANITIZED_ONNX) / (1024*1024)
    
    print(f"✅ 净化完成: {SANITIZED_ONNX} ({sanitized_size:.2f} MB)")
    print(f"净化后算子: {dict(sorted(sanitized_ops.items()))}")
    
    if 'HardSigmoid' not in sanitized_ops:
        print("🎉 所有HardSigmoid节点已成功移除！")
    else:
        print(f"❌ 仍有 {sanitized_ops['HardSigmoid']} 个HardSigmoid节点未处理")
        
except Exception as e:
    print(f"❌ ONNX净化失败: {e}")
    exit(1)

# ==================== 步骤3：更新转换脚本配置 ====================
print(f"\n⚙️  步骤3/4: 更新TensorFlow转换脚本配置...")

try:
    # 读取转换脚本
    with open(CONVERSION_SCRIPT, 'r', encoding='utf-8') as f:
        script_content = f.read()
    
    # 更新ONNX路径指向净化后的模型
    if INPUT_ONNX in script_content:
        updated_content = script_content.replace(
            f"ONNX_PATH = '{INPUT_ONNX}'",
            f"ONNX_PATH = '{SANITIZED_ONNX}'"
        )
        
        with open(CONVERSION_SCRIPT, 'w', encoding='utf-8') as f:
            f.write(updated_content)
        
        print(f"✅ 已更新转换脚本，使用净化后的模型: {SANITIZED_ONNX}")
    else:
        print(f"⚠️  转换脚本可能已经配置为使用: {SANITIZED_ONNX}")
        
except Exception as e:
    print(f"❌ 更新转换脚本失败: {e}")
    print("请手动修改convert_onnx_to_pb.py中的ONNX_PATH")

# ==================== 步骤4：执行TensorFlow转换 ====================
print(f"\n🔄 步骤4/4: 执行TensorFlow转换 (使用MTK环境)...")

try:
    # 创建在mtk-env环境中执行转换的命令
    conda_activate_cmd = f"source /home/visuals/anaconda3/etc/profile.d/conda.sh && conda activate mtk-env && python {CONVERSION_SCRIPT}"
    
    print("正在MTK环境中执行转换...")
    print(f"命令: {conda_activate_cmd}")
    
    # 使用bash执行conda命令，Python 3.6兼容版本
    result = subprocess.run(
        ["bash", "-c", conda_activate_cmd], 
        stdout=subprocess.PIPE, 
        stderr=subprocess.PIPE, 
        universal_newlines=True, 
        cwd='.'
    )
    
    print("转换脚本输出:")
    print("-" * 40)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print("错误输出:")
        print(result.stderr)
    print("-" * 40)
    
    if result.returncode == 0:
        if os.path.exists(OUTPUT_PB):
            pb_size = os.path.getsize(OUTPUT_PB) / (1024*1024)
            print(f"🎉 转换成功！")
            print(f"✅ 最终模型: {OUTPUT_PB} ({pb_size:.2f} MB)")
            
            # 显示转换总结
            print(f"\n📊 转换总结:")
            print(f"原始ONNX: {INPUT_ONNX} ({file_size:.2f} MB)")
            print(f"净化ONNX: {SANITIZED_ONNX} ({sanitized_size:.2f} MB)")
            print(f"TensorFlow PB: {OUTPUT_PB} ({pb_size:.2f} MB)")
            print(f"HardSigmoid节点处理: {processed_count}个")
            
            print(f"\n🚀 部署建议:")
            print(f"1. 使用 {OUTPUT_PB} 文件部署到MTK 8195 APU")
            print(f"2. 输入尺寸: [1, 3, 416, 416] (NCHW格式)")
            print(f"3. 预处理: 图像缩放到416x416，像素值归一化到[0,1]")
            print(f"4. 输出: [1, 2] softmax概率 [非PPT, PPT]")
            
        else:
            print(f"❌ 转换脚本执行成功但未找到输出文件 {OUTPUT_PB}")
    else:
        print(f"❌ TensorFlow转换失败 (退出码: {result.returncode})")
        
        # 分析具体错误
        if "No module named 'keras.src.engine'" in str(result.stderr):
            print("⚠️  错误原因: 环境依赖问题 - keras版本不兼容")
            print("💡 建议: 需要重新配置mtk-env环境或使用兼容的TensorFlow版本")
        elif "No schema registered for 'GatherND'" in str(result.stderr):
            print("⚠️  错误原因: ONNX模型包含onnx-tf不支持的算子")
            print("💡 建议: 模型可能需要进一步简化或使用其他转换工具")
        else:
            print("请检查onnx-tf环境和依赖是否正确安装")
        
except Exception as e:
    print(f"❌ 执行转换脚本时出错: {e}")

print("\n" + "=" * 60)
print("🏁 YOLO11模型转换流程完成")

# ==================== 备用方案：直接运行指令 ====================
print(f"\n💡 手动转换方法:")
print(f"如果自动转换失败，请在终端中手动执行:")
print(f"1. conda activate mtk-env")
print(f"2. python {CONVERSION_SCRIPT}")
print(f"\n已准备好的文件:")
print(f"- 净化后ONNX: {SANITIZED_ONNX}")
print(f"- 转换脚本: {CONVERSION_SCRIPT} (已配置使用净化后模型)")

🚀 YOLO11模型完整转换流程开始
📋 步骤1/4: 检查输入文件...
✅ 输入模型: yolo11l-model.onnx (46.70 MB)
原始模型算子: {'Conv': 10, 'Flatten': 1, 'Gemm': 1, 'GlobalAveragePool': 1, 'HardSigmoid': 10, 'Mul': 10, 'Softmax': 1}
需要处理的HardSigmoid节点: 10个

🔧 步骤2/4: ONNX净化处理 (HardSigmoid → Mul+Add+Clip)...
  处理节点 1/10: /model.0/act/HardSigmoid (α=0.1666666716337204, β=0.5)
  处理节点 2/10: /model.1/act/HardSigmoid (α=0.1666666716337204, β=0.5)
  处理节点 3/10: /model.2/act/HardSigmoid (α=0.1666666716337204, β=0.5)
  处理节点 4/10: /model.3/act/HardSigmoid (α=0.1666666716337204, β=0.5)
  处理节点 5/10: /model.4/act/HardSigmoid (α=0.1666666716337204, β=0.5)
  处理节点 6/10: /model.5/act/HardSigmoid (α=0.1666666716337204, β=0.5)
  处理节点 7/10: /model.6/act/HardSigmoid (α=0.1666666716337204, β=0.5)
  处理节点 8/10: /model.7/act/HardSigmoid (α=0.1666666716337204, β=0.5)
  处理节点 9/10: /model.8/act/HardSigmoid (α=0.1666666716337204, β=0.5)
  处理节点 10/10: /model.9/conv/act/HardSigmoid (α=0.1666666716337204, β=0.5)
  ✅ 形状推断完成
✅ 净化完成: yolo11l-model_hsigmoid2clip.on

In [9]:
import tensorflow.compat.v1 as tf
from mtk_converter import TensorFlowV1Converter
import os
import numpy as np
import cv2
import glob

tf.disable_eager_execution() # 确保在 TF 1.x 模式下运行

# --- 用户配置 ---
# 1. (!!!) 输入的 .pb 文件路径：指向你用 TF2.x 脚本生成的那个文件 (!!!)
FROZEN_PB_PATH = 'yolo11l-model_fp32_frozen.pb'

# 2. (!!!) 模型的输入输出节点名称 (!!!)
#    你的 TF2.x 脚本已经打印出了这些名称，请从那边复制粘贴过来！
#    例如: Input names: ['images'], Output names: ['output0']
INPUT_NAMES = ['images']
OUTPUT_NAMES = ['output0'] 

# 3. (!!!) 模型的输入形状 (必须是 NHWC 格式！)
INPUT_SHAPES = [[1, 416, 416, 3]] # NHWC: [批次, 高, 宽, 通道]

# 4. 你的校准图片所在的文件夹
CALIBRATION_DATA_FOLDER = '/home/visuals/Code/datasets/extracted_frames/0'

# 5. 最终输出的 TFLite 文件路径
OUTPUT_TFLITE_PATH = 'ppt_classifier_quant_final.tflite'
# --- 配置结束 ---


# --- 校准数据生成器 (确保是 NHWC 格式) ---
def calibration_dataset_generator():
    """从校准文件夹加载图片，并处理成 NHWC 格式"""
    image_files = []
    for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp']:
        image_files.extend(glob.glob(os.path.join(CALIBRATION_DATA_FOLDER, ext)))

    if not image_files:
        raise ValueError(f"在 {CALIBRATION_DATA_FOLDER} 中没有找到图片文件")

    print(f"找到 {len(image_files)} 张图片用于校准")

    for i, image_path in enumerate(image_files[:100]): # 最多使用100张
        try:
            image = cv2.imread(image_path)
            if image is None: continue

            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            image = cv2.resize(image, (416, 416))
            image = image.astype(np.float32) / 255.0
            
            # 关键：保持 NHWC 格式！
            image = np.expand_dims(image, axis=0)  # 添加批次维度 -> (1, 416, 416, 3)
            
            yield [image]
        except Exception as e:
            print(f"处理 {image_path} 时出错: {e}")
            continue
# --- 数据生成器结束 ---


print(f"正在为模型 {os.path.basename(FROZEN_PB_PATH)} 进行 PTQ 量化...")

# 1. 从冻结的 .pb 文件创建【联发科转换器】
converter = TensorFlowV1Converter.from_frozen_graph_def_file(
    graph_def_file=FROZEN_PB_PATH,
    input_names=INPUT_NAMES,
    input_shapes=INPUT_SHAPES,
    output_names=OUTPUT_NAMES
)

# 2. 设置量化参数
converter.quantize = True
converter.use_symmetric_quantization = False
converter.calibration_data_gen = calibration_dataset_generator

# 3. 执行转换
print("开始转换... 这个过程可能会持续几分钟...")
_ = converter.convert_to_tflite(output_file=OUTPUT_TFLITE_PATH)

print(f"✅ PTQ 量化和转换完成！最终模型已保存到: {OUTPUT_TFLITE_PATH}")

/home/visuals/anaconda3/envs/mtk-env/lib/python3.6/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


正在为模型 yolo11l-model_fp32_frozen.pb 进行 PTQ 量化...
找到 2276 张图片用于校准
开始转换... 这个过程可能会持续几分钟...
Importing the model ...
                                    done


KeyError: 'output0'

In [1]:
import onnx
from onnx_tf.backend import prepare
import tensorflow as tf
import os

# --- 用户配置 ---
ONNX_PATH = 'yolo11l-model.onnx'
SAVED_MODEL_PATH = 'yolo11l-model_fp32.pb'
FROZEN_PB_PATH = 'yolo11l-model_fp32_frozen.pb'

# 关键：输出节点名称，需要根据模型调整
OUTPUT_NODE_NAMES = []  # 将在运行时自动检测
# --- 配置结束 ---


# 1. 加载 ONNX 模型
print(f"Loading ONNX model from: {ONNX_PATH}")
onnx_model = onnx.load(ONNX_PATH)

# 2. 转换为 TensorFlow representation
print("Converting ONNX to TensorFlow...")
tf_rep = prepare(onnx_model, strict=False)

# 3. 导出为 SavedModel
print(f"Exporting to SavedModel: {SAVED_MODEL_PATH}")
tf_rep.export_graph(SAVED_MODEL_PATH)

# 4. 加载 SavedModel 并冻结
print("Loading SavedModel and freezing...")
model = tf.saved_model.load(SAVED_MODEL_PATH)

# 获取签名
concrete_func = model.signatures['serving_default']

# 自动检测输出节点名称
print("Detecting output node names...")
OUTPUT_NODE_NAMES = [output.name.split(':')[0] for output in concrete_func.outputs]
print(f"Detected output nodes: {OUTPUT_NODE_NAMES}")

# 获取图
graph = concrete_func.graph

# 使用 TF 1.x 风格冻结
print("Freezing graph using TF 1.x style...")
with tf.compat.v1.Session(graph=graph) as sess:
    # 初始化变量
    sess.run(tf.compat.v1.global_variables_initializer())

    # 冻结图
    frozen_graph_def = tf.compat.v1.graph_util.convert_variables_to_constants(
        sess=sess,
        input_graph_def=graph.as_graph_def(),
        output_node_names=OUTPUT_NODE_NAMES
    )

# 5. 保存冻结后的 .pb 文件
print(f"Saving frozen graph to: {FROZEN_PB_PATH}")
with tf.io.gfile.GFile(FROZEN_PB_PATH, "wb") as f:
    f.write(frozen_graph_def.SerializeToString())

print(f"\n✅ Frozen .pb model saved to: {FROZEN_PB_PATH}")
print("\n下一步建议：")
print(f"1. 使用 Netron 检查 {FROZEN_PB_PATH}")
print("2. 确认模型结构正确")
print("3. 可以使用这个 .pb 文件进行 MediaTek 量化")

ImportError: cannot import name 'dtensor'

In [ ]:
# TensorFlow 1.14 版本量化代码 - MediaTek VPU 兼容
import tensorflow as tf
import numpy as np
import cv2
import glob
import os

# 确保使用 TF 1.x API
tf.compat.v1.disable_v2_behavior()

# --- 用户配置 ---
FROZEN_PB_PATH = 'yolo11l-model_fp32_frozen.pb'
OUTPUT_TFLITE_PATH = 'ppt_classifier_quant_ptq_tf1.14.tflite'
CALIBRATION_DATA_FOLDER = '/home/visuals/Code/datasets/extracted_frames/0'

# 模型输入输出配置
INPUT_NODE_NAME = 'images'  # 根据模型调整
OUTPUT_NODE_NAMES = ['Identity']  # 根据模型调整
INPUT_SHAPE = [1, 3, 416, 416]  # NCHW 格式
# --- 配置结束 ---

print("🔧 TensorFlow 1.14 量化流程")
print(f"输入模型: {FROZEN_PB_PATH}")
print(f"输出模型: {OUTPUT_TFLITE_PATH}")
print(f"数据格式: NCHW {INPUT_SHAPE}")
print()

def representative_dataset_gen():
    """生成代表性数据集 - TF 1.14 风格"""
    def _generator():
        image_files = []
        for ext in ['*.jpg', '*.jpeg', '*.png', '*.bmp']:
            image_files.extend(glob.glob(os.path.join(CALIBRATION_DATA_FOLDER, ext)))

        print(f"找到 {len(image_files)} 张校准图片")

        # 处理前100张图片用于校准
        for i, image_path in enumerate(image_files[:100]):
            try:
                # 加载和预处理图片
                image = cv2.imread(image_path)
                if image is None:
                    continue

                # BGR 转 RGB
                image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

                # 调整大小
                image = cv2.resize(image, (416, 416))

                # 归一化到 [0, 1]
                image = image.astype(np.float32) / 255.0

                # 转换为 NCHW 格式 (模型期望的格式)
                # HWC -> CHW -> NCHW
                image = np.transpose(image, (2, 0, 1))  # (416, 416, 3) -> (3, 416, 416)
                image = np.expand_dims(image, axis=0)  # (1, 3, 416, 416)

                if i < 3:
                    print(f"处理图片 {i+1}: {os.path.basename(image_path)} - 形状: {image.shape}")

                yield image

            except Exception as e:
                print(f"处理图片错误 {image_path}: {e}")
                continue

    return _generator

# 1. 创建 TFLite 转换器
print("创建 TFLite 转换器...")
converter = tf.lite.TFLiteConverter.from_frozen_graph(
    graph_def_file=FROZEN_PB_PATH,
    input_arrays=[INPUT_NODE_NAME],
    output_arrays=OUTPUT_NODE_NAMES,
    input_shapes={INPUT_NODE_NAME: INPUT_SHAPE}
)

# 2. 设置量化参数
# converter.optimizations = [tf.lite.Optimize.DEFAULT]  # 注释掉以避免不支持的操作
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_ops = [tf.lite.OpsSet.TFLITE_BUILTINS, tf.lite.OpsSet.SELECT_TF_OPS]
# converter.inference_input_type = tf.int8  # 注释掉
# converter.inference_output_type = tf.int8  # 注释掉
# 移除量化设置，使用动态范围量化

# 3. 设置代表性数据集
print("设置代表性数据集...")
converter.representative_dataset = tf.lite.RepresentativeDataset(representative_dataset_gen())

# 4. 执行量化转换
print("执行 INT8 量化转换...")
tflite_quantized_model = converter.convert()

# 5. 保存量化模型
print(f"保存量化模型到: {OUTPUT_TFLITE_PATH}")
with open(OUTPUT_TFLITE_PATH, 'wb') as f:
    f.write(tflite_quantized_model)

print("✅ TensorFlow 1.14 量化完成!")
print(f"量化模型大小: {len(tflite_quantized_model)} 字节")
print()
print("⚠️ 重要警告:")
print("此模型使用 NCHW 格式，在 MediaTek VPU 上可能无法正常工作!")
print("MediaTek VPU 仅支持 NHWC 格式。")
print()
print("🔧 建议的解决方案:")
print("1. 在部署前将模型转换为 NHWC 格式")
print("2. 或者使用训练时就采用 NHWC 格式的模型")
print("3. 在 MediaTek 设备上测试 - 可能因格式不匹配而失败")